# Tone-Aware Review Voice Agent v2 — Colab T4 runner (Shaadi-Sahulat Integration)

This notebook runs the FastAPI app on a Colab T4 GPU, integrated with the Shaadi-Sahulat e-commerce project.

The UI opens through **Colab's built-in port proxy** — no ngrok, no Cloudflare, no signup.

**Before you start:** open `Runtime → Change runtime type` and pick **T4 GPU**.

**You will need:**
1. `tone-voice-agent-v2-colab.zip` from the project folder (upload prompt in step 3).
2. A Groq API key from https://console.groq.com/keys.

The pipeline is Groq (LLM) + Kokoro (TTS). Groq runs remotely; Kokoro runs on the T4.

**Integration with Shaadi-Sahulat:**
- The tone-voice-agent is placed inside `visual-ml-service/models/tone_voice_agent/`
- It generates voice reviews for the e-commerce comment/review section
- 4 language agents: EN Male/Female, Urdu Male/Female
- Audio files are saved with structured naming: `buyer_id_product_id_agent_hash.wav`
- Voice metadata (not actual audio blob) is stored in the Review DB model
- Audio files live in `uploads/Reviews/voices/` directory

## 1. Verify the GPU

In [ ]:
!nvidia-smi

## 2. System dependencies

Kokoro uses `espeak-ng` for phonemization and `pydub` uses `ffmpeg` for audio handling.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq espeak-ng ffmpeg
!espeak-ng --version

## 3. Upload the project zip

Run the cell below, click **Choose Files**, and pick `tone-voice-agent-v2-colab.zip` from your local machine.

The zip contains: `app.py`, `tone_*.py`, `llm_groq.py`, `tts_kokoro.py`, `cache.py`, `tone_config.json`, `requirements.txt`, and the `static/` folder.

In [ ]:
import os, zipfile, shutil, glob
from google.colab import files

PROJECT_DIR = '/content/tone-voice-agent-v2'
os.chdir('/content')

# Remove any previous uploads
for zp in glob.glob('/content/*.zip'):
    print(f'Removing previous zip: {zp}')
    os.remove(zp)

if os.path.exists(PROJECT_DIR):
    print(f'Removing previous project dir: {PROJECT_DIR}')
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

uploaded = files.upload()
zip_name = next(iter(uploaded))
print(f'Received {zip_name} ({len(uploaded[zip_name])} bytes)')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(PROJECT_DIR)

# Flatten if single top-level folder
entries = os.listdir(PROJECT_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(PROJECT_DIR, entries[0])):
    inner = os.path.join(PROJECT_DIR, entries[0])
    for item in os.listdir(inner):
        shutil.move(os.path.join(inner, item), os.path.join(PROJECT_DIR, item))
    os.rmdir(inner)

os.chdir(PROJECT_DIR)
print('Project files:')
print(sorted(os.listdir(PROJECT_DIR)))

## 4. Python dependencies

Colab already ships CUDA-enabled torch. Install kokoro, fastapi, groq, etc.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q nest-asyncio

## 5. Set your Groq API key

In [ ]:
from getpass import getpass
groq_key = getpass('Paste your GROQ_API_KEY: ').strip()
with open('.env', 'w') as f:
    f.write(f'GROQ_API_KEY={groq_key}\n')
print('.env written (key hidden).')

## 6. Verify Kokoro voice IDs

This also downloads Kokoro model weights (~350 MB) on first run.

In [ ]:
import json
from kokoro import KPipeline

with open('tone_config.json') as f:
    cfg = json.load(f)
print('Configured voice IDs:', cfg['voice_ids'])

for lang_code, label in [('a', 'English'), ('h', 'Hindi')]:
    print(f'\nLoading {label} pipeline (lang_code={lang_code!r})...')
    p = KPipeline(lang_code=lang_code)
    voices = getattr(p, 'voices', None)
    if voices is not None:
        print(f'  Available: {sorted(voices)}')
    else:
        print('  (This Kokoro version does not expose .voices; check the Kokoro repo for current IDs.)')

## 7. Launch the app and open the UI

Uvicorn runs in a background thread on port 8000.

In [ ]:
import sys, threading, time, nest_asyncio, uvicorn, requests
from google.colab import output

nest_asyncio.apply()

# Stop any previous server
_prev_server = globals().get('_UVICORN_SERVER')
_prev_thread = globals().get('_UVICORN_THREAD')
if _prev_server is not None:
    print('Stopping previous server...')
    _prev_server.should_exit = True
    if _prev_thread is not None:
        _prev_thread.join(timeout=10)
    time.sleep(1)
    print('Previous server stopped.')

# Purge cached modules for fresh imports
PROJECT_MODULES = {'app', 'tone_analyzer', 'tone_resolver', 'tone_router',
                   'tone_config', 'llm_groq', 'tts_kokoro', 'cache',
                   'tone_voice_routes'}
for m in list(sys.modules):
    if m in PROJECT_MODULES or any(m.startswith(p + '.') for p in PROJECT_MODULES):
        del sys.modules[m]

# Import fresh and launch
from app import app
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
_UVICORN_SERVER = server

def _run():
    server.run()

thread = threading.Thread(target=_run, name='uvicorn', daemon=True)
_UVICORN_THREAD = thread
thread.start()

for _ in range(60):
    time.sleep(1)
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=1).ok:
            break
    except Exception:
        pass
else:
    raise RuntimeError('Server did not start in 60 seconds.')

print('Server is up. Opening UI in a new tab...')
output.serve_kernel_port_as_window(8000, path='/')

## 8. Integration Test — Shaadi-Sahulat Review Voice

Test the tone-voice agent API integration for e-commerce reviews.

In [ ]:
import requests, json

# Test analyze endpoint
r = requests.post('http://127.0.0.1:8000/analyze', json={
    'text': 'The delivery was late and the food was cold. Really disappointed.',
    'rating': 2,
})
print('=== Analysis Result ===')
print(json.dumps(r.json(), indent=2))

# Test synthesize with buyer/product context
r2 = requests.post('http://127.0.0.1:8000/synthesize', json={
    'text': 'The delivery was late and the food was cold. Really disappointed.',
    'rating': 2,
    'agent': 'en_male',
    'buyer_id': 'BUY-001',
    'product_id': 'PRD-001',
    'order_id': 'ORD-2026-001',
})
print('\n=== Synthesis Result (with review metadata) ===')
result = r2.json()
print(json.dumps(result, indent=2))
if result.get('voice_metadata'):
    print('\nVoice metadata for review integration:')
    print(json.dumps(result['voice_metadata'], indent=2))

# Test all 4 agents
for agent in ['en_male', 'en_female', 'hi_male', 'hi_female']:
    r3 = requests.post('http://127.0.0.1:8000/synthesize', json={
        'text': 'Beautiful product, fast delivery!',
        'rating': 5,
        'agent': agent,
        'buyer_id': 'BUY-001',
        'product_id': 'PRD-002',
        'order_id': 'ORD-2026-002',
    })
    print(f'\n=== Agent: {agent} ===')
    res = r3.json()
    print(f'Audio URL: {res.get("audio_url", "N/A")}')
    if res.get('voice_metadata'):
        print(f'Voice file: {res["voice_metadata"]["voice_file"]}')

## 9. Notes

- **First synthesis is slow** — Kokoro downloads model weights (~350 MB) on first call.
- **GPU usage** — Kokoro uses T4 automatically via torch CUDA.
- **Colab limits** — Free Colab disconnects after ~90 min idle or 12 h total.
- **Hot-reload** — Edit `tone_config.json`, then `POST /reload-config`.
- **Audio files** — Generated WAVs in `/content/tone-voice-agent-v2/outputs/`.
- **Integration** — Voice metadata is returned in `/synthesize` response for storage in the e-commerce Review DB.
- **4 agents** — en_male (am_michael), en_female (af_bella), hi_male (hm_omega), hi_female (hf_beta).
- **Review integration** — Voice file naming: `{buyer_id}_{product_id}_{agent}_{hash}.wav`